In [1]:
# For tips on running notebooks in Google Colab, see
# https://docs.pytorch.org/tutorials/beginner/colab
%matplotlib inline

[Learn the Basics](intro.html) \|\|
[Quickstart](quickstart_tutorial.html) \|\|
[Tensors](tensorqs_tutorial.html) \|\| [Datasets &
DataLoaders](data_tutorial.html) \|\|
[Transforms](transforms_tutorial.html) \|\| [Build
Model](buildmodel_tutorial.html) \|\|
[Autograd](autogradqs_tutorial.html) \|\| **Optimization** \|\| [Save &
Load Model](saveloadrun_tutorial.html)

Optimizing Model Parameters
===========================

Now that we have a model and data it\'s time to train, validate and test
our model by optimizing its parameters on our data. Training a model is
an iterative process; in each iteration the model makes a guess about
the output, calculates the error in its guess (*loss*), collects the
derivatives of the error with respect to its parameters (as we saw in
the [previous section](autogradqs_tutorial.html)), and **optimizes**
these parameters using gradient descent. For a more detailed walkthrough
of this process, check out this video on [backpropagation from
3Blue1Brown](https://www.youtube.com/watch?v=tIeHLnjs5U8).

Prerequisite Code
-----------------

We load the code from the previous sections on [Datasets &
DataLoaders](data_tutorial.html) and [Build
Model](buildmodel_tutorial.html).


In [4]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor()
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor()
)

print(f"Numero de patrones de entrenamiento: {len(training_data)}")
print(f"Numero de patrones de test: {len(test_data)}")

# Crea lotes de 64 muestras para entrenamiento (baraja por defecto según configuración)
train_dataloader = DataLoader(training_data, batch_size=64)  
test_dataloader = DataLoader(test_data, batch_size=64)  # Crea lotes de 64 muestras para evaluación del modelo

print(f"Numero de batches de entrenamiento: {len(train_dataloader)}")
print(f"Numero de batches de test: {len(test_dataloader)}")

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()  # Convierte cada imagen 28x28 en un vector de 784 características
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),  # Primera capa lineal: 784 -> 512
            nn.ReLU(),  # No linealidad para aprender patrones complejos
            nn.Linear(512, 512),  # Segunda capa oculta
            nn.ReLU(),  # Nueva activación no lineal
            nn.Linear(512, 10),  # Capa de salida: 10 logits (uno por clase)
        )

    def forward(self, x):
        x = self.flatten(x)  # Aplana el lote de imágenes antes de capas lineales
        logits = self.linear_relu_stack(x)  # Calcula puntuaciones crudas por clase
        return logits  # Devuelve logits; la función de pérdida aplicará la transformación necesaria

model = NeuralNetwork()  # Instancia del modelo listo para entrenar

Numero de patrones de entrenamiento: 60000
Numero de patrones de test: 10000
Numero de batches de entrenamiento: 938
Numero de batches de test: 157


Hyperparameters
===============

Hyperparameters are adjustable parameters that let you control the model
optimization process. Different hyperparameter values can impact model
training and convergence rates ([read
more](https://pytorch.org/tutorials/beginner/hyperparameter_tuning_tutorial.html)
about hyperparameter tuning)

We define the following hyperparameters for training:

:   -   **Number of Epochs** - the number of times to iterate over the
        dataset
    -   **Batch Size** - the number of data samples propagated through
        the network before the parameters are updated
    -   **Learning Rate** - how much to update models parameters at each
        batch/epoch. Smaller values yield slow learning speed, while
        large values may result in unpredictable behavior during
        training.


In [5]:
learning_rate = 1e-3
batch_size = 64
epochs = 5

Optimization Loop
=================

Once we set our hyperparameters, we can then train and optimize our
model with an optimization loop. Each iteration of the optimization loop
is called an **epoch**.

Each epoch consists of two main parts:

:   -   **The Train Loop** - iterate over the training dataset and try
        to converge to optimal parameters.
    -   **The Validation/Test Loop** - iterate over the test dataset to
        check if model performance is improving.

Let\'s briefly familiarize ourselves with some of the concepts used in
the training loop. Jump ahead to see the
`full-impl-label`{.interpreted-text role="ref"} of the optimization
loop.

Loss Function
-------------

When presented with some training data, our untrained network is likely
not to give the correct answer. **Loss function** measures the degree of
dissimilarity of obtained result to the target value, and it is the loss
function that we want to minimize during training. To calculate the loss
we make a prediction using the inputs of our given data sample and
compare it against the true data label value.

Common loss functions include
[nn.MSELoss](https://pytorch.org/docs/stable/generated/torch.nn.MSELoss.html#torch.nn.MSELoss)
(Mean Square Error) for regression tasks, and
[nn.NLLLoss](https://pytorch.org/docs/stable/generated/torch.nn.NLLLoss.html#torch.nn.NLLLoss)
(Negative Log Likelihood) for classification.
[nn.CrossEntropyLoss](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html#torch.nn.CrossEntropyLoss)
combines `nn.LogSoftmax` and `nn.NLLLoss`.

We pass our model\'s output logits to `nn.CrossEntropyLoss`, which will
normalize the logits and compute the prediction error.


In [6]:
# Initialize the loss function
loss_fn = nn.CrossEntropyLoss()

Optimizer
=========

Optimization is the process of adjusting model parameters to reduce
model error in each training step. **Optimization algorithms** define
how this process is performed (in this example we use Stochastic
Gradient Descent). All optimization logic is encapsulated in the
`optimizer` object. Here, we use the SGD optimizer; additionally, there
are many [different
optimizers](https://pytorch.org/docs/stable/optim.html) available in
PyTorch such as ADAM and RMSProp, that work better for different kinds
of models and data.

We initialize the optimizer by registering the model\'s parameters that
need to be trained, and passing in the learning rate hyperparameter.


In [7]:
# model es una instancia del modelo de rede neuronal definido e instanciado previamente
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

Inside the training loop, optimization happens in three steps:

:   -   Call `optimizer.zero_grad()` to reset the gradients of model
        parameters. Gradients by default add up; to prevent
        double-counting, we explicitly zero them at each iteration.
    -   Backpropagate the prediction loss with a call to
        `loss.backward()`. PyTorch deposits the gradients of the loss
        w.r.t. each parameter.
    -   Once we have our gradients, we call `optimizer.step()` to adjust
        the parameters by the gradients collected in the backward pass.


Full Implementation {#full-impl-label}
===================

We define `train_loop` that loops over our optimization code, and
`test_loop` that evaluates the model\'s performance against our test
data.


In [8]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)  # Número total de muestras de entrenamiento
    # Set the model to training mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.train()  # Activa modo entrenamiento (afecta capas como Dropout/BatchNorm)
    for batch, (X, y) in enumerate(dataloader):
        # Compute prediction and loss
        pred = model(X)  # Forward pass: predicciones (logits) para el lote actual
        loss = loss_fn(pred, y)  # Calcula la pérdida comparando predicción vs etiqueta real

        # Backpropagation
        loss.backward()  # Calcula gradientes de la pérdida respecto a los parámetros
        optimizer.step()  # Actualiza parámetros usando los gradientes calculados
        optimizer.zero_grad()  # Limpia gradientes acumulados para la siguiente iteración

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)  # `current`: muestras procesadas hasta este punto
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")  # Muestra progreso de entrenamiento


def test_loop(dataloader, model, loss_fn):
    # Set the model to evaluation mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.eval()  # Activa modo evaluación (desactiva comportamiento estocástico)
    size = len(dataloader.dataset)  # Número total de muestras de test
    num_batches = len(dataloader)  # Número de lotes de test
    test_loss, correct = 0, 0

    # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():  # Evita construir grafo y ahorra memoria/tiempo en inferencia
        for X, y in dataloader:
            pred = model(X)  # Predicción del lote de test
            test_loss += loss_fn(pred, y).item()  # Acumula pérdida media por lote
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()  # Cuenta aciertos del lote

    test_loss /= num_batches  # Pérdida promedio en todo el conjunto de test
    correct /= size  # Proporción de acierto total (accuracy en [0,1])
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")  # Reporte final de evaluación

We initialize the loss function and optimizer, and pass it to
`train_loop` and `test_loop`. Feel free to increase the number of epochs
to track the model\'s improving performance.


In [9]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.303252  [   64/60000]
loss: 2.289838  [ 6464/60000]
loss: 2.269292  [12864/60000]
loss: 2.259189  [19264/60000]
loss: 2.235579  [25664/60000]
loss: 2.207942  [32064/60000]
loss: 2.215288  [38464/60000]
loss: 2.186023  [44864/60000]
loss: 2.183875  [51264/60000]
loss: 2.143715  [57664/60000]
Test Error: 
 Accuracy: 46.4%, Avg loss: 2.141849 

Epoch 2
-------------------------------
loss: 2.152826  [   64/60000]
loss: 2.146453  [ 6464/60000]
loss: 2.084252  [12864/60000]
loss: 2.101779  [19264/60000]
loss: 2.047448  [25664/60000]
loss: 1.986080  [32064/60000]
loss: 2.012729  [38464/60000]
loss: 1.939444  [44864/60000]
loss: 1.945702  [51264/60000]
loss: 1.868088  [57664/60000]
Test Error: 
 Accuracy: 57.0%, Avg loss: 1.870134 

Epoch 3
-------------------------------
loss: 1.901046  [   64/60000]
loss: 1.877086  [ 6464/60000]
loss: 1.756738  [12864/60000]
loss: 1.801066  [19264/60000]
loss: 1.690052  [25664/60000]
loss: 1.644570  [32064/600

Further Reading
===============

-   [Loss
    Functions](https://pytorch.org/docs/stable/nn.html#loss-functions)
-   [torch.optim](https://pytorch.org/docs/stable/optim.html)
-   [Warmstart Training a
    Model](https://pytorch.org/tutorials/recipes/recipes/warmstarting_model_using_parameters_from_a_different_model.html)
